# Notebook 14E — Fannie Mae predictor compatibility
Audit the original 29 Freddie predictors using the reviewed diagnostic sample. This notebook does not create model-ready features, fit preprocessing, train or score models. Candidate mappings are not final semantic approvals.

References: [Fannie glossary (July 2024)](https://capitalmarkets.fanniemae.com/sites/capmrkt/files/2024-07/crt-file-layout-and-glossary.pdf), [Freddie user guide](https://www.freddiemac.com/fmac-resources/research/pdf/user_guide.pdf), [Freddie FAQ](https://www.freddiemac.com/fmac-resources/research/pdf/faq.pdf), and local `src/freddie_features.py`. Exact local Fannie release vintage remains unverified.

## 1. Read the frozen Freddie contract
Verify the saved baseline manifest and feature contract before examining Fannie data.

In [1]:
from pathlib import Path
import ast, hashlib, json, re, zipfile
from collections import Counter
from functools import lru_cache
from datetime import datetime, timezone
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()
SOURCE = ROOT/'data/raw/Fannie_Mae/2018Q1/2018Q1.csv'
BASE = ROOT/'data/processed/Freddie_Mac/model_baselines_2015_2017'
FEATURE_SOURCE = ROOT/'src/freddie_features.py'
manifest = json.loads((BASE/'model_manifest.json').read_text(encoding='utf-8'))
contract_file = BASE/'freddie_feature_contract.json'
contract = json.loads(contract_file.read_text(encoding='utf-8'))
sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
assert sha(contract_file) == manifest['feature_contract_sha256']
assert manifest['horizon_months'] == 24
constants = {t.id: ast.literal_eval(node.value) for node in ast.parse(FEATURE_SOURCE.read_text(encoding='utf-8')).body if isinstance(node,ast.Assign) for t in node.targets if isinstance(t,ast.Name) and t.id in {'NUMERIC_FEATURES','CATEGORICAL_FEATURES'}}
NUM = constants['NUMERIC_FEATURES']; CAT = constants['CATEGORICAL_FEATURES']
assert NUM+CAT == manifest['predictors'] and len(NUM+CAT)==29
assert set(NUM)==set(contract['numeric_medians']) and set(CAT)==set(contract['category_levels'])
assert contract['training_years']==[2015,2016,2017]
print('Verified frozen contract:', len(NUM), 'numeric and', len(CAT), 'categorical predictors')

Verified frozen contract: 15 numeric and 14 categorical predictors


## 2. Recreate the same loan sample
Use only the previously inspected origination fields and reporting month. High-balance and program positions are not assumed. Scan the complete file to capture all records of selected loans.

In [2]:
SIZE = SOURCE.stat().st_size
WIDTH = 113
assert SIZE == 6928164686, 'Source differs from reviewed file.'
POSITIONS = {'loan_id':2,'report':3,'channel':4,'rate':8,'upb':10,'term':13,'orig':14,'first_payment':15,'ltv':20,'cltv':21,'borrowers':22,'dti':23,'borrower_fico':24,'coborrower_fico':25,'fthb':26,'purpose':27,'property_type':28,'units':29,'occupancy':30,'state':31,'mi':34,'amortization':35,'interest_only':37}
SAMPLE_MODULUS = 100
MAX_SAMPLE_ROWS = 1_000_000
SOURCE_SIGNATURE = (SOURCE.stat().st_size, SOURCE.stat().st_mtime_ns)
@lru_cache(maxsize=100_000)
def selected(loan_id):
    return int.from_bytes(hashlib.blake2b(loan_id,digest_size=8,person=b'Fannie14v1').digest(),'big') % SAMPLE_MODULUS == 0
def scan_source(path, expected_width, progress_every=2_000_000):
    counts = Counter()
    sampled = []
    widths_seen = Counter()
    id_pattern = re.compile(rb'[A-Za-z0-9]{1,12}')
    indexes = [p-1 for p in POSITIONS.values()]
    with path.open('rb') as stream:
        for line_number, raw in enumerate(stream, 1):
            parts = raw.rstrip(b'\r\n').split(b'|')
            if line_number == 1:
                parts[0] = parts[0].removeprefix(b'\xef\xbb\xbf')
            counts['physical_rows'] += 1
            widths_seen[len(parts)] += 1
            if len(parts) != expected_width:
                counts['wrong_width'] += 1
                continue
            loan_id = parts[1].strip()
            if not id_pattern.fullmatch(loan_id):
                counts['invalid_loan_id'] += 1
                continue
            counts['valid_id_rows'] += 1
            if selected(loan_id):
                sampled.append(tuple(parts[i].strip().decode('ascii') for i in indexes))
                if len(sampled) > MAX_SAMPLE_ROWS:
                    raise RuntimeError('Sample exceeds memory guard; increase SAMPLE_MODULUS and rerun the entire scan.')
            if progress_every and line_number % progress_every == 0:
                print(f'{line_number:,} rows scanned; {len(sampled):,} sampled rows')
    return pd.DataFrame(sampled, columns=list(POSITIONS)), counts, widths_seen

history, scan_counts, full_widths = scan_source(SOURCE, WIDTH)
assert SOURCE_SIGNATURE == (SOURCE.stat().st_size, SOURCE.stat().st_mtime_ns), 'Source changed during scan.'
print(dict(scan_counts), dict(full_widths))
assert not scan_counts['wrong_width'] and not scan_counts['invalid_loan_id'], 'Malformed records found; review before continuing.'
assert not history.empty, 'No sampled rows'
print('Sample rows:', len(history), '| Sample distinct loans:', history.loan_id.nunique())
assert scan_counts['physical_rows'] == 23563399
assert len(history) == 245552 and history.loan_id.nunique() == 4749, 'Sample differs from reviewed Stage 2.' 

2,000,000 rows scanned; 22,660 sampled rows
4,000,000 rows scanned; 43,404 sampled rows
6,000,000 rows scanned; 63,865 sampled rows
8,000,000 rows scanned; 83,683 sampled rows
10,000,000 rows scanned; 103,966 sampled rows
12,000,000 rows scanned; 123,590 sampled rows
14,000,000 rows scanned; 145,624 sampled rows
16,000,000 rows scanned; 165,802 sampled rows
18,000,000 rows scanned; 186,285 sampled rows
20,000,000 rows scanned; 206,376 sampled rows
22,000,000 rows scanned; 229,575 sampled rows
{'physical_rows': 23563399, 'valid_id_rows': 23563399} {113: 23563399}
Sample rows: 245552 | Sample distinct loans: 4749


## 3. Audit values at loan level
Use the earliest reporting row only as a diagnostic snapshot, not as a certified origination record. Count changes over the full history, including transitions between blank and populated values. Do not backfill from later months.

In [3]:
assert history.report.str.fullmatch(r'(0[1-9]|1[0-2])(19|20)\d{2}').all()
history['report_idx'] = history.report.str[2:].astype(int)*12 + history.report.str[:2].astype(int)
assert not history.duplicated(['loan_id','report_idx']).any()
first = history.sort_values('report_idx').drop_duplicates('loan_id').copy()
fields = [f for f in POSITIONS if f not in {'loan_id','report'}]
profiles=[]; categories=[]
for field in fields:
    v=first[field]; present=v.ne(''); numeric=pd.to_numeric(v.where(present),errors='coerce')
    distinct=history.groupby('loan_id')[field].nunique(dropna=False)
    nonblank_distinct=history[field].where(history[field].ne('')).groupby(history.loan_id).nunique()
    profiles.append(dict(field=field,physical_position=POSITIONS[field],loans=len(first),blank=int((~present).sum()),nonblank=int(present.sum()),numeric_parseable=int(numeric.notna().sum()),numeric_min=numeric.min(),numeric_max=numeric.max(),loans_with_any_change=int(distinct.gt(1).sum()),loans_with_nonblank_conflict=int(nonblank_distinct.gt(1).sum())))
    if field in ['channel','term','borrowers','fthb','purpose','property_type','units','occupancy','state','amortization','interest_only']:
        for value,count in v.value_counts().items(): categories.append(dict(field=field,value=value or '[blank]',loans=int(count)))
profile=pd.DataFrame(profiles); category_counts=pd.DataFrame(categories)
display(profile)

,field,physical_position,loans,blank,nonblank,numeric_parseable,numeric_min,numeric_max,loans_with_any_change,loans_with_nonblank_conflict
0,channel,4,4749,0,4749,0,NaN,NaN,1,1
1,rate,8,4749,0,4749,4749,2.75,6.125,0,0
2,upb,10,4749,0,4749,4749,24000.00,1223000.000,0,0
3,term,13,4749,0,4749,4749,96.00,360.000,0,0
4,orig,14,4749,0,4749,4749,12017.00,122017.000,4,4
5,first_payment,15,4749,0,4749,4749,12018.00,122017.000,1,1
6,ltv,20,4749,0,4749,4749,9.00,97.000,0,0
7,cltv,21,4749,0,4749,4749,9.00,105.000,0,0
8,borrowers,22,4749,0,4749,4749,1.00,4.000,0,0
9,dti,23,4749,1,4748,4748,1.00,50.000,1,0


## 4. Record candidate mappings for all 29 predictors
Keep questionable definitions unresolved. A missing field is not equivalent to an observed zero or a negative flag. Derived predictors inherit the limitations of their source fields.

In [4]:
# predictor: (Fannie source fields, candidate status, issue or transformation)
M = {
'FICO_SCORE':('borrower_fico;coborrower_fico;borrowers','requires_investigation','Verify Freddie loan-level representative score versus Fannie borrower-level scores; do not blindly take minimum.'),
'MI_PERCENT':('mi','requires_investigation','Confirm blank versus no insurance versus unavailable and comparable coverage definition.'),
'ORIGINAL_DTI':('dti','direct_candidate','Verify units, valid ranges and missing codes.'),
'ORIGINAL_LTV':('ltv','direct_candidate','Verify units, valid ranges and missing codes.'),
'ORIGINAL_INTEREST_RATE':('rate','direct_candidate','Original rate in percentage points; verify definition.'),
'LOG_ORIGINAL_UPB':('upb','harmonization_candidate','Use log1p of original UPB, never current UPB.'),
'CLTV_MINUS_LTV':('cltv;ltv','harmonization_candidate','Use max(CLTV-LTV,0) after comparable cleaning.'),
'HAS_MORTGAGE_INSURANCE':('mi','requires_investigation','Depends on MI blank semantics; Freddie uses MI > 0.'),
'HAS_SUBORDINATE_FINANCING':('cltv;ltv','harmonization_candidate','Freddie uses CLTV > LTV; verify missingness rules.'),
'HIGH_LTV_GT_80':('ltv','harmonization_candidate','Freddie uses LTV > 80.'),
'FICO_SCORE_MISSING':('borrower_fico;coborrower_fico;borrowers','requires_investigation','Depends on verified score definition.'),
'MI_PERCENT_MISSING':('mi','requires_investigation','Depends on MI blank semantics.'),
'ORIGINAL_DTI_MISSING':('dti','harmonization_candidate','Match documented missing codes before flagging.'),
'ORIGINAL_LTV_MISSING':('ltv','harmonization_candidate','Match documented missing codes before flagging.'),
'CLTV_MISSING':('cltv','harmonization_candidate','Match documented missing codes before flagging.'),
'FIRST_TIME_HOMEBUYER':('fthb','harmonization_candidate','Confirm unknown codes and borrower definition.'),
'NUMBER_OF_UNITS':('units','direct_candidate','Check category values and missing codes.'),
'OCCUPANCY_STATUS':('occupancy','harmonization_candidate','Compare primary, second and investment categories.'),
'CHANNEL':('channel','harmonization_candidate','Compare retail, broker and correspondent definitions.'),
'AMORTIZATION_TYPE':('amortization','direct_candidate','Fannie population excludes ARM; verify structural restriction.'),
'PROPERTY_STATE':('state','direct_candidate','Check state codes.'),
'PROPERTY_TYPE':('property_type','harmonization_candidate','Compare property type category definitions.'),
'LOAN_PURPOSE':('purpose','harmonization_candidate','Check cash-out and non-cash-out refinance definitions; map codes explicitly.'),
'ORIGINAL_LOAN_TERM':('term','harmonization_candidate','Treat term in months as categorical.'),
'NUMBER_OF_BORROWERS_GROUP':('borrowers','harmonization_candidate','Group 1 vs 2+ after validating count and unknowns.'),
'SUPER_CONFORMING_FLAG':('','requires_investigation','Fannie high-balance indicator is a potential analogue; verify position, applicability and limits.'),
'SPECIAL_ELIGIBILITY_PROGRAM':('','requires_investigation','Program names and coding may differ; no verified equivalent yet.'),
'HARP_INDICATOR':('','requires_investigation','Primary Fannie dataset excludes HARP; verify population and whether a structural constant is justified.'),
'INTEREST_ONLY_INDICATOR':('interest_only','requires_investigation','Fannie population excludes interest-only loans; verify applicability of field and blanks.')}
assert set(M)==set(NUM+CAT)
mapping=pd.DataFrame([dict(predictor=p,source_fields=M[p][0],candidate_status=M[p][1],review_issue=M[p][2],final_approved=False) for p in NUM+CAT])
assert all(f in fields for s in mapping.source_fields for f in s.split(';') if f)
display(mapping); print(mapping.candidate_status.value_counts())

,predictor,source_fields,candidate_status,review_issue,final_approved
0,FICO_SCORE,borrower_fico;coborrower_fico;borrowers,requires_investigation,Verify Freddie loan-level representative score...,False
1,MI_PERCENT,mi,requires_investigation,Confirm blank versus no insurance versus unava...,False
2,ORIGINAL_DTI,dti,direct_candidate,"Verify units, valid ranges and missing codes.",False
3,ORIGINAL_LTV,ltv,direct_candidate,"Verify units, valid ranges and missing codes.",False
4,ORIGINAL_INTEREST_RATE,rate,direct_candidate,Original rate in percentage points; verify def...,False
5,LOG_ORIGINAL_UPB,upb,harmonization_candidate,"Use log1p of original UPB, never current UPB.",False
6,CLTV_MINUS_LTV,cltv;ltv,harmonization_candidate,"Use max(CLTV-LTV,0) after comparable cleaning.",False
7,HAS_MORTGAGE_INSURANCE,mi,requires_investigation,Depends on MI blank semantics; Freddie uses MI...,False
8,HAS_SUBORDINATE_FINANCING,cltv;ltv,harmonization_candidate,Freddie uses CLTV > LTV; verify missingness ru...,False
9,HIGH_LTV_GT_80,ltv,harmonization_candidate,Freddie uses LTV > 80.,False


candidate_status
harmonization_candidate    14
requires_investigation      9
direct_candidate            6
Name: count, dtype: int64


## 5. Compare categorical values with frozen Freddie levels
This is a literal code overlap check, not a harmonization decision. A different code may represent the same concept, and matching codes may have different definitions.

In [5]:
overlap=[]
for p in CAT:
    f=M[p][0]
    if not f or ';' in f: continue
    allowed=set(map(str,contract['category_levels'][p]))
    for value,count in first[f].value_counts().items():
        overlap.append(dict(predictor=p,source_field=f,fannie_raw_value=value or '[blank]',loans=int(count),literal_match_in_frozen_levels=value in allowed,frozen_levels=';'.join(sorted(allowed))))
overlap=pd.DataFrame(overlap)
# Additional diagnostics for the two most consequential unresolved numeric mappings.
fico_mi=first.assign(borrower_group=first.borrowers.replace('', '[blank]'),borrower_score_blank=first.borrower_fico.eq(''),coborrower_score_blank=first.coborrower_fico.eq(''),mi_blank=first.mi.eq(''))
score_counts=fico_mi.groupby(['borrower_group','borrower_score_blank','coborrower_score_blank'],dropna=False).size().reset_index(name='loans')
mi_counts=first.assign(ltv_band=pd.cut(pd.to_numeric(first.ltv,errors='coerce'),[-float('inf'),80,float('inf')],labels=['<=80','>80']).astype('string').fillna('unknown'),mi_state=first.mi.map(lambda v:'blank' if not v else 'numeric zero' if re.fullmatch(r'0+(\.0+)?',v) else 'other nonblank')).groupby(['ltv_band','mi_state']).size().reset_index(name='loans')
display(score_counts); display(mi_counts)

,borrower_group,borrower_score_blank,coborrower_score_blank,loans
0,1,False,True,2588
1,1,True,True,4
2,2,False,False,2113
3,2,False,True,6
4,3,False,False,31
5,4,False,False,7


,ltv_band,mi_state,loans
0,<=80,blank,3375
1,<=80,other nonblank,1
2,>80,blank,13
3,>80,other nonblank,1360


## 6. Export aggregate evidence
No loan identifiers, loan-level data, labels or model files are exported. Review the provisional mappings before selecting a common feature contract.

In [6]:
out=ROOT/'data/processed/Fannie_Mae/notebook_14e_predictor_compatibility'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
out.mkdir(parents=True,exist_ok=False)
for name,table in {'predictor_mapping.csv':mapping,'source_field_profile.csv':profile,'source_category_counts.csv':category_counts,'frozen_category_overlap.csv':overlap,'credit_score_availability.csv':score_counts,'mi_availability_by_ltv.csv':mi_counts}.items(): table.to_csv(out/name,index=False)
meta=dict(stage='14E',source_name=SOURCE.name,file_bytes=SIZE,physical_rows=int(scan_counts['physical_rows']),sample_rows=len(history),sample_loans=len(first),sample_modulus=SAMPLE_MODULUS,feature_source_sha256=sha(FEATURE_SOURCE),frozen_contract_sha256=sha(contract_file),predictor_count=len(mapping),final_approved_predictors=0,release_vintage='unverified',models_scored=False,features_constructed=False,scope='all sampled loans; earliest reporting snapshot; no eligibility or outcome selection')
(out/'metadata.json').write_text(json.dumps(meta,indent=2),encoding='utf-8')
review=out/'notebook_14e_predictor_compatibility_review.zip'
with zipfile.ZipFile(review,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(out.iterdir()):
        if p!=review: z.write(p,p.name)
print('Review ZIP:',review)

Review ZIP: e:\capstone_project\mortgage-model-reliability\data\processed\Fannie_Mae\notebook_14e_predictor_compatibility\20261124T153634575349Z\notebook_14e_predictor_compatibility_review.zip
